In [1]:
# ============================================================
# 15_naked_atm_put_daily_production_v0_1
# Daily production notebook for naked ATM put v0.1
# ============================================================

import json
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime

cwd = Path.cwd()

if cwd.name.lower() == "notebooks":
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
AUDIT_DIR = PROJECT_ROOT / "data" / "audit"
PRODUCTION_DIR = PROJECT_ROOT / "data" / "production"

PRODUCTION_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PROCESSED_DATA_DIR:", PROCESSED_DATA_DIR)
print("AUDIT_DIR:", AUDIT_DIR)
print("PRODUCTION_DIR:", PRODUCTION_DIR)

PROJECT_ROOT: C:\Users\patri\vrp_project
PROCESSED_DATA_DIR: C:\Users\patri\vrp_project\data\processed
AUDIT_DIR: C:\Users\patri\vrp_project\data\audit
PRODUCTION_DIR: C:\Users\patri\vrp_project\data\production


In [2]:
# ============================================================
# Load frozen production candidate
# ============================================================

PRODUCTION_SPEC_JSON = (
    AUDIT_DIR / "naked_atm_put_production_candidate_v0_1_spec.json"
)

SIZED_TRADES_PARQUET = (
    PROCESSED_DATA_DIR / "naked_atm_put_tenor_sizing_trades_v0_1.parquet"
)

assert PRODUCTION_SPEC_JSON.exists(), PRODUCTION_SPEC_JSON
assert SIZED_TRADES_PARQUET.exists(), SIZED_TRADES_PARQUET

with open(PRODUCTION_SPEC_JSON, "r") as f:
    production_spec = json.load(f)

sized_trades = pd.read_parquet(SIZED_TRADES_PARQUET).copy()

sized_trades["trade_dt"] = pd.to_datetime(sized_trades["trade_dt"])
sized_trades["entry_tenor"] = sized_trades["entry_tenor"].astype(int)

production_trades = (
    sized_trades[
        sized_trades["strategy_name"].eq("primary_with_refined_hybrid_override")
        & sized_trades["sizing_scheme"].eq("tenor_plus_override_sizeup")
    ]
    .copy()
    .sort_values("trade_dt")
    .reset_index(drop=True)
)

print("Production candidate:")
print(production_spec["candidate_name"])

print("\nRows:", len(production_trades))
print("Start:", production_trades["trade_dt"].min())
print("End:", production_trades["trade_dt"].max())

display(production_trades.tail())

Production candidate:
naked_atm_put_production_candidate_v0_1

Rows: 602
Start: 2020-10-29 00:00:00
End: 2026-06-05 00:00:00


,trade_date,tenor,spx_close,spx_rsi_14,vix_style_vol,implied_variance,old_vrp_signal,old_vol,old_vrp_3m_z,old_vrp_1y_z,old_vrp_blended_z,simple_forecast_model,simple_forecast_variance,forecast_vol,forecast_vrp_signal,forecast_vrp_3m_z,forecast_vrp_1y_z,forecast_vrp_blended_z,target_tenor,side,source_universe,trade_date_ts,target_expiry_date,entry_spx_close,tenor_group,trade_year,expiry_mapping_status,selected_expiry_date,selected_chain_file,expiry_distance_days,actual_dte,strike_mapping_status,short_strike_selected,raw_short_strike,put_moneyness,put_pct_otm,entry_quote_status,entry_bid,entry_ask,entry_mid,entry_spread,entry_spread_pct_mid,mapping_status,expiry_spx_close,expiry_spx_status,expiry_intrinsic_value,expired_otm,entry_credit_points,short_option_pnl_points,dollars_per_contract,contracts_for_normalized_notional,normalized_pnl_dollars,normalized_pnl_pct,win,pricing_status,trade_dt,entry_tenor,tenor_champion_panel,champion_forecast_vol,champion_forecast_vrp_signal,champion_forecast_vrp_signal_z_3m,champion_forecast_vrp_signal_z_1y,champion_forecast_vrp_min_z,simple_carry_vrp_signal,trailing_carry_vrp_signal,entry_rsi_14,test_pnl,test_pnl_source_col,test_win,simple_carry_vrp_signal_z_3m,simple_carry_vrp_signal_z_1y,simple_carry_vrp_min_z,trailing_carry_vrp_signal_z_3m,trailing_carry_vrp_signal_z_1y,trailing_carry_vrp_min_z,simple_minus_champion_signal,both_simple_and_champion_rich_flag,simple_rich_champion_weak_flag,champion_rich_simple_weak_flag,tenor_denom,vrp_signal_har_rv_iv,vrp_signal_har_rv_iv_min_z,pred_vol_pct_har_rv_iv,vrp_signal_har_rv_iv_per_tenor,vrp_signal_har_rv_iv_per_tenor_min_z,pred_vol_pct_har_rv_iv_per_tenor,vrp_signal_hybrid_log_objective,vrp_signal_hybrid_log_objective_min_z,pred_vol_pct_hybrid_log_objective,vrp_signal_hybrid_back_har_only,vrp_signal_hybrid_back_har_only_min_z,pred_vol_pct_hybrid_back_har_only,score,rule_id,rule_name,denominator_model,shortlist_label,rsi_cap,old_signal_threshold,old_z_threshold,simple_signal_threshold,simple_z_threshold,denom_signal_threshold,denom_z_threshold,priority,combo_strategy,is_refined_override,strategy_name,sizing_scheme,size_multiplier,sized_pnl,primary_entry_tenor,primary_tenor_group
597,20260326,30,6477.16,33.164094,28.248942,0.079800,1.272550,14.951042,0.736434,0.814857,0.775646,ewma_10_scheduled_raw,0.021811,14.768428,1.297129,0.929709,1.087601,1.008655,30,put,all_date_tenor_atm_put,2026-03-26,20260425,6477.16,back_27_33d,2026,mapped,20260424,C:\Users\patri\vrp_project\data\raw\thetadata_...,-1,29,mapped,6475.0,6477.0,0.999667,0.000333,mapped,164.0,167.9,165.95,3.9,0.023501,mapped,7165.08,mapped,0.00,True,164.0,164.00,16400.0,1.543887,25319.738898,0.025320,True,priced,2026-03-26,30,30.0,22.925773,0.417589,-0.309499,-0.513670,-0.513670,1.297129,1.272550,33.164094,25319.738898,normalized_pnl_dollars,True,0.905031,1.079056,0.905031,0.714633,0.808311,0.714633,0.879540,True,False,False,30.0,0.417589,-0.513670,22.925773,0.432815,-0.362784,22.751897,0.417589,-0.513670,22.925773,1.217714,1.096670,15.366639,1.629866,1729.0,old_simple_confirm_reference,none_reference,primary_old_simple_robust,72,0.5,-0.5,0.4,0.0,NaN,NaN,1,primary_with_refined_hybrid_override,False,primary_with_refined_hybrid_override,tenor_plus_override_sizeup,1.15,29117.699733,30.0,back_27_33d
598,20260327,33,6368.85,28.658202,30.857478,0.095218,1.324787,15.910598,0.812669,0.858471,0.835570,ewma_10_scheduled_raw,0.024984,15.806303,1.337941,0.993221,1.093591,1.043406,33,put,all_date_tenor_atm_put,2026-03-27,20260429,6368.85,back_27_33d,2026,mapped,20260501,C:\Users\patri\vrp_project\data\raw\thetadata_...,2,35,mapped,6360.0,6368.0,0.998610,0.001390,mapped,193.5,194.3,193.90,0.8,0.004126,mapped,7230.12,mapped,0.00,True,193.5,193.50,19350.0,1.570142,30382.251113,0.030382,True,priced,2026-03-27,33,33.0,24.205921,0.485564,0.421792,-0.144823,-0.144823,1.337941,1.324787,28.658202,30382.251113,normalized_pnl_dollars,True,0.966436,1.084884,0.966436,0.791072,0.851543,0.791072,0.852377,True,False,False,33.

In [3]:
# ============================================================
# Production parameters
# ============================================================

# Use None to select the latest available trade date in the candidate file.
# Or set manually, e.g. RUN_DATE = "2026-06-05"
RUN_DATE = None

# Base size is your normal naked ATM put risk unit.
# For now this is unit-based. Later we can convert to dollars/contracts.
BASE_SIZE_UNITS = 1.00

# Optional execution inputs.
# Fill these in manually when moving from recommendation to actual trade ticket.
ATM_PUT_STRIKE = None
ATM_PUT_MID = None
CONTRACT_MULTIPLIER = 100
BASE_NOTIONAL = None  # e.g. 100_000, if using dollar notional sizing

if RUN_DATE is None:
    run_dt = production_trades["trade_dt"].max()
else:
    run_dt = pd.Timestamp(RUN_DATE)

print("RUN_DATE:", run_dt.date())
print("BASE_SIZE_UNITS:", BASE_SIZE_UNITS)

RUN_DATE: 2026-06-05
BASE_SIZE_UNITS: 1.0


In [4]:
# ============================================================
# Generate daily recommendation
# ============================================================

daily_rows = production_trades[
    production_trades["trade_dt"].eq(run_dt)
].copy()

if daily_rows.empty:
    daily_recommendation = pd.DataFrame([
        {
            "run_dt": run_dt,
            "trade_signal": False,
            "action": "NO TRADE",
            "reason": "No production candidate signal for this date.",
        }
    ])
else:
    assert len(daily_rows) == 1, f"Expected 1 trade row, found {len(daily_rows)}"

    row = daily_rows.iloc[0]

    size_multiplier = float(row["size_multiplier"])
    target_size_units = BASE_SIZE_UNITS * size_multiplier

    if BASE_NOTIONAL is not None:
        target_notional = BASE_NOTIONAL * size_multiplier
    else:
        target_notional = None

    if (
        target_notional is not None
        and ATM_PUT_STRIKE is not None
        and CONTRACT_MULTIPLIER is not None
    ):
        contracts = int(np.floor(target_notional / (ATM_PUT_STRIKE * CONTRACT_MULTIPLIER)))
    else:
        contracts = None

    daily_recommendation = pd.DataFrame([
        {
            "run_dt": run_dt,
            "trade_signal": True,
            "action": "SELL ATM PUT",
            "selected_tenor": int(row["entry_tenor"]),
            "tenor_group": row["tenor_group"],
            "is_refined_override": bool(row.get("is_refined_override", False)),
            "primary_entry_tenor": row.get("primary_entry_tenor", np.nan),
            "size_multiplier": size_multiplier,
            "base_size_units": BASE_SIZE_UNITS,
            "target_size_units": target_size_units,
            "base_notional": BASE_NOTIONAL,
            "target_notional": target_notional,
            "atm_put_strike": ATM_PUT_STRIKE,
            "atm_put_mid": ATM_PUT_MID,
            "contract_multiplier": CONTRACT_MULTIPLIER,
            "contracts": contracts,
            "model_strategy": "primary_with_refined_hybrid_override",
            "sizing_scheme": "tenor_plus_override_sizeup",
        }
    ])

display(daily_recommendation)

,run_dt,trade_signal,action,selected_tenor,tenor_group,is_refined_override,primary_entry_tenor,size_multiplier,base_size_units,target_size_units,base_notional,target_notional,atm_put_strike,atm_put_mid,contract_multiplier,contracts,model_strategy,sizing_scheme
0,2026-06-05,True,SELL ATM PUT,15,front_9_15d,False,15.0,0.75,1.0,0.75,None,None,None,None,100,None,primary_with_refined_hybrid_override,tenor_plus_override_sizeup


In [5]:
# ============================================================
# Save daily recommendation
# ============================================================

run_date_str = pd.Timestamp(run_dt).strftime("%Y%m%d")

DAILY_RECOMMENDATION_CSV = (
    PRODUCTION_DIR / f"naked_atm_put_daily_recommendation_{run_date_str}_v0_1.csv"
)

daily_recommendation.to_csv(DAILY_RECOMMENDATION_CSV, index=False)

print("Saved daily recommendation:")
print(DAILY_RECOMMENDATION_CSV)

Saved daily recommendation:
C:\Users\patri\vrp_project\data\production\naked_atm_put_daily_recommendation_20260605_v0_1.csv


In [6]:
# ============================================================
# Append to production recommendation log
# ============================================================

RECOMMENDATION_LOG_CSV = (
    PRODUCTION_DIR / "naked_atm_put_recommendation_log_v0_1.csv"
)

log_row = daily_recommendation.copy()
log_row["created_at"] = pd.Timestamp.now()

if RECOMMENDATION_LOG_CSV.exists():
    existing_log = pd.read_csv(RECOMMENDATION_LOG_CSV)
    existing_log["run_dt"] = pd.to_datetime(existing_log["run_dt"])

    # Remove existing row for this run date to avoid duplicate reruns.
    existing_log = existing_log[
        ~existing_log["run_dt"].eq(pd.Timestamp(run_dt))
    ]

    updated_log = pd.concat([existing_log, log_row], ignore_index=True)
else:
    updated_log = log_row.copy()

updated_log = updated_log.sort_values("run_dt").reset_index(drop=True)
updated_log.to_csv(RECOMMENDATION_LOG_CSV, index=False)

print("Updated recommendation log:")
print(RECOMMENDATION_LOG_CSV)

display(updated_log.tail(10))

Updated recommendation log:
C:\Users\patri\vrp_project\data\production\naked_atm_put_recommendation_log_v0_1.csv


,run_dt,trade_signal,action,selected_tenor,tenor_group,is_refined_override,primary_entry_tenor,size_multiplier,base_size_units,target_size_units,base_notional,target_notional,atm_put_strike,atm_put_mid,contract_multiplier,contracts,model_strategy,sizing_scheme,created_at
0,2026-06-05,True,SELL ATM PUT,15,front_9_15d,False,15.0,0.75,1.0,0.75,None,None,None,None,100,None,primary_with_refined_hybrid_override,tenor_plus_override_sizeup,2026-07-01 19:24:49.284810
